## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import pickle
import contextlib
import io

from kama_msr import KAMA_MSR
from kmrf import KMRF
from bayesian_forward_simulator import BayesianForwardSimulator
from portfolio_optimizer_inputs import PortfolioOptimizerInputs
from portfolio_optimizer import PortfolioOptimizer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [2]:
# Test configuration
TEST_ASSET_CLASS = 'us_equity'
TEST_ASSET_NAME = 'SPDR S&P 500 ETF'
TEST_END_DATE = '20241101'  # Recent date for testing

# Color scheme for pass/fail
PASS = '\033[92m✓ PASS\033[0m'
FAIL = '\033[91m✗ FAIL\033[0m'
WARN = '\033[93m⚠ WARNING\033[0m'

def print_test(name, passed, message=''):
    status = PASS if passed else FAIL
    print(f"{status} - {name}")
    if message:
        print(f"     {message}")
    return passed

def print_section(title):
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}")

---
## Test 1: Date Alignment Validation

In [3]:
print_section("TEST 1: DATE ALIGNMENT")

# Load master data to check available date range
master_df = pd.read_csv('data/master_df.csv', index_col=0, header=[0, 1, 2], parse_dates=True)
master_label_df = pd.read_csv('data/master_label_df.csv', index_col=0, header=[0, 1], parse_dates=True)

print(f"\nMaster Data Date Range:")
print(f"  Features: {master_df.index[0]} to {master_df.index[-1]}")
print(f"  Labels:   {master_label_df.index[0]} to {master_label_df.index[-1]}")

test_end = pd.Timestamp(TEST_END_DATE)
print(f"\nTest End Date: {test_end}")

# Check if test date is within available data
test1a = print_test(
    "Test date within feature range",
    test_end <= master_df.index[-1],
    f"Features available up to {master_df.index[-1]}"
)

test1b = print_test(
    "Labels may not extend to test date (expected for recent dates)",
    master_label_df.index[-1] > test_end,
    f"Labels only to {master_label_df.index[-1]} - will need KAMA_MSR model"
)

# Check for sufficient OOS data after test date
dates_after = master_df.index[master_df.index > test_end]
test1c = print_test(
    "Sufficient OOS data for validation/test split",
    len(dates_after) >= 42,  # At least 2 months for val+test
    f"{len(dates_after)} trading days available after {test_end}"
)

print(f"\n{WARN} - For dates after {master_label_df.index[-1]}, must use retrain_kmrf=True with KAMA_MSR models")


TEST 1: DATE ALIGNMENT

Master Data Date Range:
  Features: 1990-01-02 00:00:00 to 2025-10-31 00:00:00
  Labels:   1995-01-30 00:00:00 to 2025-10-07 00:00:00

Test End Date: 2024-11-01 00:00:00
✓ PASS - Test date within feature range
     Features available up to 2025-10-31 00:00:00
✓ PASS - Labels may not extend to test date (expected for recent dates)
     Labels only to 2025-10-07 00:00:00 - will need KAMA_MSR model
✓ PASS - Sufficient OOS data for validation/test split
     259 trading days available after 2024-11-01 00:00:00

⚠ WARNING - For dates after 2025-10-07 00:00:00, must use retrain_kmrf=True with KAMA_MSR models

Master Data Date Range:
  Features: 1990-01-02 00:00:00 to 2025-10-31 00:00:00
  Labels:   1995-01-30 00:00:00 to 2025-10-07 00:00:00

Test End Date: 2024-11-01 00:00:00
✓ PASS - Test date within feature range
     Features available up to 2025-10-31 00:00:00
✓ PASS - Labels may not extend to test date (expected for recent dates)
     Labels only to 2025-10-07 0

---
## Test 2: KAMA_MSR Label Loading

In [4]:
print_section("TEST 2: KAMA_MSR LABEL LOADING")

# Check if KAMA_MSR model exists for test date
kama_msr_path = Path('saved_models') / 'KAMA_MSR' / TEST_ASSET_CLASS / TEST_END_DATE / f"{TEST_ASSET_NAME}_KAMA-MSR_4-regimes.pkl"

test2a = print_test(
    f"KAMA_MSR model exists for {TEST_END_DATE}",
    kama_msr_path.exists(),
    f"Path: {kama_msr_path}"
)

if kama_msr_path.exists():
    # Load and check labels
    with open(kama_msr_path, 'rb') as f:
        kama_msr = pickle.load(f)
    
    labels = kama_msr.regime_labels
    print(f"\nKAMA_MSR Labels:")
    print(f"  Date range: {labels.index[0]} to {labels.index[-1]}")
    print(f"  Total samples: {len(labels)}")
    
    test2b = print_test(
        "Labels extend to end_date",
        labels.index[-1] >= pd.Timestamp(TEST_END_DATE),
        f"Last label: {labels.index[-1]}, end_date: {TEST_END_DATE}"
    )
    
    # Check regime distribution
    print(f"\n  Regime Distribution:")
    regime_names = {0: 'LV Bull', 1: 'LV Bear', 2: 'HV Bull', 3: 'HV Bear'}
    for regime in [0, 1, 2, 3]:
        count = (labels == regime).sum()
        pct = (count / len(labels)) * 100
        print(f"    {regime_names[regime]:>8}: {count:>5} ({pct:>5.1f}%)")
    
    test2c = print_test(
        "All regimes represented",
        all((labels == r).sum() > 0 for r in [0, 1, 2, 3]),
        "All 4 regimes have at least one sample"
    )
else:
    print(f"\n{FAIL} - Cannot proceed with label validation")


TEST 2: KAMA_MSR LABEL LOADING
✓ PASS - KAMA_MSR model exists for 20241101
     Path: saved_models/KAMA_MSR/us_equity/20241101/SPDR S&P 500 ETF_KAMA-MSR_4-regimes.pkl

KAMA_MSR Labels:
  Date range: 1995-01-04 00:00:00 to 2024-11-01 00:00:00
  Total samples: 7511
✓ PASS - Labels extend to end_date
     Last label: 2024-11-01 00:00:00, end_date: 20241101

  Regime Distribution:
     LV Bull:  5629 ( 74.9%)
     LV Bear:  1366 ( 18.2%)
     HV Bull:    65 (  0.9%)
     HV Bear:   433 (  5.8%)
✓ PASS - All regimes represented
     All 4 regimes have at least one sample


---
## Test 3: KMRF Training with Retrain

In [5]:
print_section("TEST 3: KMRF TRAINING (RETRAIN=TRUE)")

print(f"\nInitializing KMRF with retrain...")
print(f"  Asset: {TEST_ASSET_NAME}")
print(f"  End Date: {TEST_END_DATE}")

# Initialize KMRF
kmrf = KMRF(
    asset_name=TEST_ASSET_NAME,
    asset_class=TEST_ASSET_CLASS,
    end_date=TEST_END_DATE,
    classification_type='original',
    use_data_type='master',
    random_seed=1010
)

# Run pipeline (suppress output)
print(f"\nRunning KMRF pipeline...")
with contextlib.redirect_stdout(io.StringIO()):
    kmrf.pipeline()

print(f"\nData Split Summary:")
print(f"  Training:   {kmrf.X_train.shape} - {kmrf.X_train.index[0]} to {kmrf.X_train.index[-1]}")
print(f"  Labels:     {kmrf.y_train.shape} - {kmrf.y_train.index[0]} to {kmrf.y_train.index[-1]}")
print(f"  Validation: {kmrf.X_val.shape} - {kmrf.X_val.index[0]} to {kmrf.X_val.index[-1]}")
print(f"  Test:       {kmrf.X_test.shape} - {kmrf.X_test.index[0]} to {kmrf.X_test.index[-1]}")

# Validation tests
test3a = print_test(
    "Training features end at or near end_date",
    abs((kmrf.X_train.index[-1] - pd.Timestamp(TEST_END_DATE)).days) <= 5,
    f"Train end: {kmrf.X_train.index[-1]}, end_date: {TEST_END_DATE}"
)

test3b = print_test(
    "Labels available for training period",
    len(kmrf.y_train) > 0 and len(kmrf.y_train) == len(kmrf.X_train),
    f"X_train: {len(kmrf.X_train)}, y_train: {len(kmrf.y_train)}"
)

test3c = print_test(
    "Validation set is non-empty",
    len(kmrf.X_val) > 0,
    f"Validation samples: {len(kmrf.X_val)}"
)

test3d = print_test(
    "Test set is non-empty",
    len(kmrf.X_test) > 0,
    f"Test samples: {len(kmrf.X_test)}"
)

test3e = print_test(
    "Val and Test sets are approximately equal",
    abs(len(kmrf.X_val) - len(kmrf.X_test)) <= 1,
    f"Val: {len(kmrf.X_val)}, Test: {len(kmrf.X_test)}"
)

test3f = print_test(
    "Model is trained (xgb_model exists)",
    kmrf.xgb_model is not None,
    "XGBoost model successfully trained"
)


TEST 3: KMRF TRAINING (RETRAIN=TRUE)

Initializing KMRF with retrain...
  Asset: SPDR S&P 500 ETF
  End Date: 20241101
KMRF model initialized
  Asset: SPDR S&P 500 ETF
  Asset class: us_equity
  Classification type: original
  Training end date: 20241101
  Test end date: All available data
  Data type: master
  Data path: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/data/master_df.csv
  KAMA+MSR model directory: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/saved_models/KAMA_MSR/us_equity/20241101
  Validation/Test split: Will be calculated after loading data
  Random seed: 1010
  Feature window size: 1 days
  Feature asset classes: []
  Feature selection: Boruta=False, Consensus=False

Running KMRF pipeline...

Data Split Summary:
  Training:   (7241, 122) - 1996-01-29 00:00:00 to 2024-11-01 00:00:00
  Labels:     (7241,) - 1996-01-29 00:00:00 to 2024-11-01 00:00:00
  Validation: (124, 122) - 2024-11-04 00:00:00 to 2025-05-05 00:00:00


---
## Test 4: KMRF OOS Predictions

In [6]:
print_section("TEST 4: KMRF OOS PREDICTIONS")

# Test predict_all_oos()
print(f"\nGenerating all OOS predictions...")
all_oos_preds = kmrf.predict_all_oos()

print(f"\nOOS Predictions:")
print(f"  Shape: {all_oos_preds.shape}")
print(f"  Date range: {all_oos_preds.index[0]} to {all_oos_preds.index[-1]}")
print(f"  Columns: {list(all_oos_preds.columns)}")

test4a = print_test(
    "OOS predictions start after end_date",
    all_oos_preds.index[0] > pd.Timestamp(TEST_END_DATE),
    f"First prediction: {all_oos_preds.index[0]}, end_date: {TEST_END_DATE}"
)

test4b = print_test(
    "OOS predictions span val+test period",
    len(all_oos_preds) == len(kmrf.X_val) + len(kmrf.X_test),
    f"OOS: {len(all_oos_preds)}, Val+Test: {len(kmrf.X_val) + len(kmrf.X_test)}"
)

test4c = print_test(
    "Probabilities sum to 1.0",
    np.allclose(all_oos_preds.sum(axis=1), 1.0),
    f"Sum range: [{all_oos_preds.sum(axis=1).min():.6f}, {all_oos_preds.sum(axis=1).max():.6f}]"
)

# Test accessing specific date
next_day = all_oos_preds.index[0]
print(f"\nTesting specific date access:")
print(f"  Target date: {next_day}")
pred_at_date = all_oos_preds.loc[next_day]
print(f"  Prediction: {dict(pred_at_date)}")

test4d = print_test(
    "Can access prediction at specific date",
    pred_at_date is not None and len(pred_at_date) == 4,
    f"Retrieved {len(pred_at_date)} regime probabilities"
)


TEST 4: KMRF OOS PREDICTIONS

Generating all OOS predictions...

Generating predictions for all out-of-sample data...
  Input shape: (249, 122)
  Date range: 2024-11-04 00:00:00 to 2025-10-31 00:00:00
✓ Generated predictions for all OOS data: (249, 4)
  Date range: 2024-11-04 00:00:00 to 2025-10-31 00:00:00

OOS Predictions:
  Shape: (249, 4)
  Date range: 2024-11-04 00:00:00 to 2025-10-31 00:00:00
  Columns: ['P(LV_Bull)', 'P(LV_Bear)', 'P(HV_Bull)', 'P(HV_Bear)']
✓ PASS - OOS predictions start after end_date
     First prediction: 2024-11-04 00:00:00, end_date: 20241101
✓ PASS - OOS predictions span val+test period
     OOS: 249, Val+Test: 249
✓ PASS - Probabilities sum to 1.0
     Sum range: [1.000000, 1.000000]

Testing specific date access:
  Target date: 2024-11-04 00:00:00
  Prediction: {'P(LV_Bull)': np.float32(0.99829537), 'P(LV_Bear)': np.float32(0.00025885587), 'P(HV_Bull)': np.float32(0.0013248359), 'P(HV_Bear)': np.float32(0.00012095684)}
✓ PASS - Can access prediction at

---
## Test 5: Bayesian Forward Simulation

In [7]:
print_section("TEST 5: BAYESIAN FORWARD SIMULATION")

# Reload KAMA_MSR for test (to ensure clean state)
with open(kama_msr_path, 'rb') as f:
    kama_msr_test = pickle.load(f)

print(f"\nKAMA_MSR regime labels end: {kama_msr_test.regime_labels.index[-1]}")
print(f"KMRF end_date: {kmrf.end_date}")

# Create simulator
simulator = BayesianForwardSimulator(
    kama_msr=kama_msr_test,
    kmrf=kmrf,
    n_days=21,
    alpha_confidence=0.75,
    significance_level=0.05
)

test5a = print_test(
    "Simulator initialized",
    simulator is not None,
    f"Forward simulation horizon: {simulator.n_days} days"
)

# Compute forward regime probabilities
print(f"\nComputing forward regime probabilities...")
with contextlib.redirect_stdout(io.StringIO()):
    simulator.compute_forward_regime_probs()

print(f"\nForward Regime Probabilities:")
print(f"  Shape: {simulator.forward_probs.shape}")
print(f"\nDay 1 (next trading day):")
print(simulator.forward_probs.iloc[0])

test5b = print_test(
    "Forward probabilities computed",
    simulator.forward_probs is not None,
    f"Shape: {simulator.forward_probs.shape}"
)

test5c = print_test(
    "Forward probs sum to 1.0 each day",
    np.allclose(simulator.forward_probs[[0,1,2,3]].sum(axis=1), 1.0),
    f"Sum range: [{simulator.forward_probs.sum(axis=1).min():.6f}, {simulator.forward_probs.sum(axis=1).max():.6f}]"
)

# Run simulations
print(f"\nRunning return simulations (1000 paths)...")
with contextlib.redirect_stdout(io.StringIO()):
    simulator.fit_regime_distributions(verbose=False)
    sims = simulator.simulate(n_simulations=1000, random_seed=1010)

print(f"\nSimulated Returns:")
print(f"  Shape: {sims.shape}")
print(f"  Mean: {sims.mean().mean():.6f}")
print(f"  Std:  {sims.std().mean():.6f}")

test5d = print_test(
    "Simulations completed",
    sims is not None and sims.shape == (21, 1000),
    f"Generated {sims.shape[1]} paths of {sims.shape[0]} days"
)

test5e = print_test(
    "Simulations have no NaN values",
    not sims.isna().any().any(),
    "All simulated returns are valid"
)


TEST 5: BAYESIAN FORWARD SIMULATION

KAMA_MSR regime labels end: 2024-11-01 00:00:00
KMRF end_date: 20241101
✓ PASS - Simulator initialized
     Forward simulation horizon: 21 days

Computing forward regime probabilities...

Forward Regime Probabilities:
  Shape: (21, 5)

Day 1 (next trading day):
regime
0                     0.983254
1                     0.011223
2                     0.001974
3                     0.003549
uncertainty_factor    0.039021
Name: 2024-11-04 00:00:00, dtype: float64
✓ PASS - Forward probabilities computed
     Shape: (21, 5)
✓ PASS - Forward probs sum to 1.0 each day
     Sum range: [1.039021, 1.366119]

Running return simulations (1000 paths)...

Forward Regime Probabilities:
  Shape: (21, 5)

Day 1 (next trading day):
regime
0                     0.983254
1                     0.011223
2                     0.001974
3                     0.003549
uncertainty_factor    0.039021
Name: 2024-11-04 00:00:00, dtype: float64
✓ PASS - Forward probabilities co

---
## Test 6: Multi-Asset Portfolio Inputs

In [8]:
print_section("TEST 6: MULTI-ASSET PORTFOLIO INPUTS")

# Test with 3 assets
test_assets = [
    'SPDR S&P 500 ETF',
    'iShares Russell 2000 ETF',
    'Invesco QQQ Trust'
]

print(f"\nGenerating portfolio inputs for {len(test_assets)} assets...")
print(f"Assets: {test_assets}")

results = PortfolioOptimizerInputs.quick_run(
    asset_names=test_assets,
    asset_class=TEST_ASSET_CLASS,
    end_date=TEST_END_DATE,
    n_days=21,
    n_simulations=1000,  # Smaller for speed
    method='path_covariance',
    annualize=True,
    verbose=False,
    random_seed=1010,
    retrain_kmrf=True,
    use_boruta_selection=False,
    use_consensus_selection=False
)

mu = results['inputs']['mu']
Sigma = results['inputs']['Sigma']

print(f"\nExpected Returns (μ):")
print(mu)

print(f"\nCovariance Matrix (Σ):")
print(Sigma)

test6a = print_test(
    "Returns vector has correct shape",
    mu.shape == (len(test_assets),),
    f"Shape: {mu.shape}"
)

test6b = print_test(
    "Covariance matrix has correct shape",
    Sigma.shape == (len(test_assets), len(test_assets)),
    f"Shape: {Sigma.shape}"
)

test6c = print_test(
    "Covariance matrix is symmetric",
    np.allclose(Sigma, Sigma.T),
    "Matrix is symmetric"
)

# Check positive definiteness
eigenvalues = np.linalg.eigvals(Sigma.values)
test6d = print_test(
    "Covariance matrix is positive definite",
    np.all(eigenvalues > 0),
    f"Min eigenvalue: {eigenvalues.min():.6e}"
)

test6e = print_test(
    "No NaN values in inputs",
    not mu.isna().any() and not Sigma.isna().any().any(),
    "All values are valid"
)

# Check asset_simulations are stored
test6f = print_test(
    "Asset simulations stored for Sortino",
    'asset_simulations' in results,
    f"Simulations shape: {(len(results['asset_simulations'].keys()), 
                          results['asset_simulations'][list(results['asset_simulations'].keys())[0]].shape)
                           if 'asset_simulations' in results else 'N/A'}"
)


TEST 6: MULTI-ASSET PORTFOLIO INPUTS

Generating portfolio inputs for 3 assets...
Assets: ['SPDR S&P 500 ETF', 'iShares Russell 2000 ETF', 'Invesco QQQ Trust']
KMRF model initialized
  Asset: SPDR S&P 500 ETF
  Asset class: us_equity
  Classification type: original
  Training end date: 20241101
  Test end date: All available data
  Data type: master
  Data path: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/data/master_df.csv
  KAMA+MSR model directory: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/saved_models/KAMA_MSR/us_equity/20241101
  Validation/Test split: Will be calculated after loading data
  Random seed: 1010
  Feature window size: 1 days
  Feature asset classes: []
  Feature selection: Boruta=False, Consensus=False

DISTRIBUTION PARAMETER VALIDATION
 regime distribution  valid       loc    scale       df       a        b  n_obs
      0 norminvgauss   True  0.001879 0.009602      NaN 1.32518 -0.18537   5629
      1       norma

---
## Test 7: Portfolio Optimization

In [9]:
print_section("TEST 7: PORTFOLIO OPTIMIZATION")

print(f"\nTesting Max Sharpe optimization...")
opt_sharpe = PortfolioOptimizer.from_optimizer_inputs(
    results,
    objective='max_sharpe',
    allow_short=False
)
weights_sharpe = opt_sharpe.optimize()

print(f"\nMax Sharpe Weights:")
print(weights_sharpe)

test7a = print_test(
    "Max Sharpe: Weights sum to 1.0",
    np.isclose(weights_sharpe.sum(), 1.0),
    f"Sum: {weights_sharpe.sum():.6f}"
)

test7b = print_test(
    "Max Sharpe: Long-only constraint satisfied",
    np.all(weights_sharpe >= -1e-6),
    f"Min weight: {weights_sharpe.min():.6f}"
)

test7c = print_test(
    "Max Sharpe: Portfolio return is positive",
    opt_sharpe.portfolio_return > 0,
    f"Return: {opt_sharpe.portfolio_return:.4f}"
)

test7d = print_test(
    "Max Sharpe: Portfolio volatility is positive",
    opt_sharpe.portfolio_risk > 0,
    f"Volatility: {opt_sharpe.portfolio_risk:.4f}"
)

# Test with short selling
print(f"\nTesting Max Sharpe with short selling (1.5x gross exposure)...")
opt_short = PortfolioOptimizer.from_optimizer_inputs(
    results,
    objective='max_sharpe',
    allow_short=True,
    gross_exposure=1.5
)
weights_short = opt_short.optimize()

print(f"\nShort Selling Weights:")
print(weights_short)

test7e = print_test(
    "Short: Weights sum to 1.0",
    np.isclose(weights_short.sum(), 1.0),
    f"Sum: {weights_short.sum():.6f}"
)

test7f = print_test(
    "Short: Gross exposure constraint satisfied",
    weights_short.abs().sum() <= 1.5 + 1e-6,
    f"Gross exposure: {weights_short.abs().sum():.6f} <= 1.5"
)

# Test Sortino optimization
print(f"\nTesting Max Sortino optimization...")
opt_sortino = PortfolioOptimizer.from_optimizer_inputs(
    results,
    objective='max_sortino',
    allow_short=False
)
weights_sortino = opt_sortino.optimize()

print(f"\nMax Sortino Weights:")
print(weights_sortino)

test7g = print_test(
    "Max Sortino: Optimization succeeded",
    weights_sortino is not None and len(weights_sortino) == len(test_assets),
    f"Generated {len(weights_sortino)} weights"
)


TEST 7: PORTFOLIO OPTIMIZATION

Testing Max Sharpe optimization...

Max Sharpe Weights:
SPDR S&P 500 ETF            0.457095
iShares Russell 2000 ETF    0.098549
Invesco QQQ Trust           0.444356
dtype: float64
✓ PASS - Max Sharpe: Weights sum to 1.0
     Sum: 1.000000
✓ PASS - Max Sharpe: Long-only constraint satisfied
     Min weight: 0.098549
✓ PASS - Max Sharpe: Portfolio return is positive
     Return: 0.1516
✓ PASS - Max Sharpe: Portfolio volatility is positive
     Volatility: 0.0985

Testing Max Sharpe with short selling (1.5x gross exposure)...

Short Selling Weights:
SPDR S&P 500 ETF            0.457095
iShares Russell 2000 ETF    0.098549
Invesco QQQ Trust           0.444356
dtype: float64
✓ PASS - Short: Weights sum to 1.0
     Sum: 1.000000
✓ PASS - Short: Gross exposure constraint satisfied
     Gross exposure: 1.000000 <= 1.5

Testing Max Sortino optimization...

Max Sortino Weights:
SPDR S&P 500 ETF            0.455172
iShares Russell 2000 ETF    0.105053
Invesco QQ

---
## Test 8: End-to-End Workflow with Multiple Dates

In [10]:
# print_section("TEST 8: END-TO-END MULTI-DATE WORKFLOW")

# # Load ETF data to get rebalance dates
# etf_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)
# etf_close = etf_data.xs('close', level=1, axis=1)
# spy_close = etf_close['SPY']

# # Get recent rebalance dates (monthly, last 3 months)
# test_rebal_dates = spy_close.loc['2025-08-07':].index[::21][:3].map(lambda x: x.strftime('%Y%m%d')).tolist()

# print(f"\nTesting workflow with {len(test_rebal_dates)} rebalance dates:")
# for date in test_rebal_dates:
#     print(f"  {date}")

# portfolio_returns = []
# optimization_errors = []

# for i, rebal_date in enumerate(test_rebal_dates):
#     print(f"\n{'─'*80}")
#     print(f"Rebalance {i+1}/{len(test_rebal_dates)}: {rebal_date}")
#     print(f"{'─'*80}")
    
#     try:
#         # Generate inputs
#         opt_inputs = PortfolioOptimizerInputs.quick_run(
#             asset_names=test_assets,
#             asset_class=TEST_ASSET_CLASS,
#             end_date=rebal_date,
#             n_days=21,
#             n_simulations=500,  # Smaller for speed
#             method='path_covariance',
#             annualize=True,
#             verbose=False,
#             random_seed=1010,
#             retrain_kmrf=True,
#             use_boruta_selection=False,
#             use_consensus_selection=False
#         )
        
#         # Optimize
#         opt = PortfolioOptimizer.from_optimizer_inputs(
#             opt_inputs,
#             objective='max_sharpe',
#             allow_short=False
#         )
#         weights = opt.optimize()
        
#         print(f"✓ Success")
#         print(f"  Portfolio Return: {opt.portfolio_return:.4f}")
#         print(f"  Portfolio Vol:    {opt.portfolio_volatility:.4f}")
#         print(f"  Sharpe Ratio:     {opt.portfolio_return / opt.portfolio_volatility:.4f}")
        
#         portfolio_returns.append({
#             'date': rebal_date,
#             'return': opt.portfolio_return,
#             'volatility': opt.portfolio_volatility,
#             'weights': weights
#         })
        
#     except Exception as e:
#         print(f"✗ Error: {str(e)}")
#         optimization_errors.append({
#             'date': rebal_date,
#             'error': str(e)
#         })

# print(f"\n{'='*80}")
# print(f"End-to-End Results")
# print(f"{'='*80}")

# test8a = print_test(
#     "All rebalance dates processed successfully",
#     len(optimization_errors) == 0,
#     f"Successes: {len(portfolio_returns)}/{len(test_rebal_dates)}"
# )

# if len(optimization_errors) > 0:
#     print(f"\nErrors encountered:")
#     for err in optimization_errors:
#         print(f"  {err['date']}: {err['error']}")

# if len(portfolio_returns) > 0:
#     print(f"\nPortfolio Performance Summary:")
#     perf_df = pd.DataFrame([{
#         'Date': r['date'],
#         'Return': f"{r['return']:.4f}",
#         'Volatility': f"{r['volatility']:.4f}",
#         'Sharpe': f"{r['return']/r['volatility']:.4f}"
#     } for r in portfolio_returns])
#     print(perf_df.to_string(index=False))

---
## Test Summary

In [11]:
print_section("VALIDATION SUMMARY")

print(f"\n{'Test Category':<40} {'Status'}")
print(f"{'─'*60}")
print(f"{'1. Date Alignment':<40} {'✓ PASS' if test1a and test1c else '✗ FAIL'}")
print(f"{'2. KAMA_MSR Labels':<40} {'✓ PASS' if test2a and test2b and test2c else '✗ FAIL'}")
print(f"{'3. KMRF Training':<40} {'✓ PASS' if all([test3a, test3b, test3c, test3d, test3e, test3f]) else '✗ FAIL'}")
print(f"{'4. KMRF Predictions':<40} {'✓ PASS' if all([test4a, test4b, test4c, test4d]) else '✗ FAIL'}")
print(f"{'5. Bayesian Simulation':<40} {'✓ PASS' if all([test5a, test5b, test5c, test5d, test5e]) else '✗ FAIL'}")
print(f"{'6. Portfolio Inputs':<40} {'✓ PASS' if all([test6a, test6b, test6c, test6d, test6e, test6f]) else '✗ FAIL'}")
print(f"{'7. Portfolio Optimization':<40} {'✓ PASS' if all([test7a, test7b, test7c, test7d, test7e, test7f, test7g]) else '✗ FAIL'}")
# print(f"{'8. End-to-End Workflow':<40} {'✓ PASS' if test8a else '✗ FAIL'}")

all_tests_passed = all([
    test1a, test1c,
    test2a, test2b, test2c,
    test3a, test3b, test3c, test3d, test3e, test3f,
    test4a, test4b, test4c, test4d,
    test5a, test5b, test5c, test5d, test5e,
    test6a, test6b, test6c, test6d, test6e, 
    # test6f,
    test7a, test7b, test7c, test7d, test7e, test7f, test7g,
    # test8a
])

print(f"\n{'='*60}")
if all_tests_passed:
    print(f"\n{PASS} - ALL VALIDATION TESTS PASSED")
    print(f"\nThe workflow is ready for production use.")
else:
    print(f"\n{FAIL} - SOME TESTS FAILED")
    print(f"\nPlease review failed tests and address issues before proceeding.")
print(f"\n{'='*60}")


VALIDATION SUMMARY

Test Category                            Status
────────────────────────────────────────────────────────────
1. Date Alignment                        ✓ PASS
2. KAMA_MSR Labels                       ✓ PASS
3. KMRF Training                         ✓ PASS
4. KMRF Predictions                      ✓ PASS
5. Bayesian Simulation                   ✓ PASS
6. Portfolio Inputs                      ✓ PASS
7. Portfolio Optimization                ✓ PASS


✓ PASS - ALL VALIDATION TESTS PASSED

The workflow is ready for production use.

